In [1]:
# Cell 1: Imports and config

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.ensemble import IsolationForest

from xgboost import XGBClassifier

RANDOM_STATE = 42


In [2]:
# Cell 2: Load dataset

df = pd.read_csv("datasets/dataset2_non_text_transactions.csv")
print("Shape:", df.shape)
df.head()


Shape: (5000, 9)


,transaction_id,transaction_date,amount,currency,vendor_id,gst_applicable,gst_slab,itc_eligible,is_anomaly
0,NTX0000001,01-04-2024,2016.36,INR,V0033,True,5%,FALSE,0
1,NTX0000002,01-04-2024,8530.60,INR,V0005,True,18%,TRUE,0
2,NTX0000003,01-04-2024,33181.64,INR,V0014,True,18%,TRUE,0
3,NTX0000004,01-04-2024,14510.68,INR,V0004,True,18%,TRUE,0
4,NTX0000005,01-04-2024,4667.38,INR,V0021,True,5%,FALSE,0


In [3]:
# Cell 3: Basic data cleaning

# Drop exact duplicates
df = df.drop_duplicates()
print("After drop_duplicates:", df.shape)

# Keep only INR
if "currency" in df.columns:
    df = df[df["currency"].astype(str).str.upper() == "INR"].copy()
    print("After INR filter:", df.shape)

# Ensure amount is numeric and non-null
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["amount"])
print("After amount numeric + non-null:", df.shape)

# Parse transaction_date (dd-mm-YYYY)
df["transaction_date"] = pd.to_datetime(df["transaction_date"], format="%d-%m-%Y", errors="coerce")
df = df.dropna(subset=["transaction_date"])
print("After valid transaction_date:", df.shape)

# Normalize gst_slab text (strip spaces, lowercase)
df["gst_slab"] = df["gst_slab"].astype(str).str.strip()

# Treat 'unknown' gst_slab as missing, drop for supervised training
df = df[df["gst_slab"].str.lower() != "unknown"].copy()

# Map 'exempt' to '0%' so slab classes are consistent
df["gst_slab"] = df["gst_slab"].replace({"exempt": "0%"})

print("Unique gst_slab values:", df["gst_slab"].unique())

# Handle vendor_id missing
if "vendor_id" in df.columns:
    df["vendor_id"] = df["vendor_id"].fillna("UNKNOWN_VENDOR")

df.head()



After drop_duplicates: (5000, 9)
After INR filter: (5000, 9)
After amount numeric + non-null: (5000, 9)
After valid transaction_date: (5000, 9)
Unique gst_slab values: ['5%' '18%' '0%' '12%' '28%']


,transaction_id,transaction_date,amount,currency,vendor_id,gst_applicable,gst_slab,itc_eligible,is_anomaly
0,NTX0000001,2024-04-01,2016.36,INR,V0033,True,5%,FALSE,0
1,NTX0000002,2024-04-01,8530.60,INR,V0005,True,18%,TRUE,0
2,NTX0000003,2024-04-01,33181.64,INR,V0014,True,18%,TRUE,0
3,NTX0000004,2024-04-01,14510.68,INR,V0004,True,18%,TRUE,0
4,NTX0000005,2024-04-01,4667.38,INR,V0021,True,5%,FALSE,0


In [4]:
# Cell 4: Feature engineering

df_feat = df.copy()

# Date features
dt = df_feat["transaction_date"]
df_feat["txn_month"] = dt.dt.month
df_feat["txn_quarter"] = dt.dt.quarter
df_feat["txn_dow"] = dt.dt.dayofweek
df_feat["financial_year"] = np.where(dt.dt.month >= 4, dt.dt.year, dt.dt.year - 1)

# Amount features
df_feat["amount_log"] = np.log1p(df_feat["amount"])

bins = [0, 1000, 5000, 20000, np.inf]
labels = ["0-1k", "1k-5k", "5k-20k", "20k+"]
df_feat["amount_bucket"] = pd.cut(df_feat["amount"], bins=bins, labels=labels, include_lowest=True)

# Vendor features
if "vendor_id" in df_feat.columns:
    vc = df_feat["vendor_id"].value_counts()
    df_feat["vendor_txn_count"] = df_feat["vendor_id"].map(vc).fillna(1)
    df_feat["vendor_txn_count_log"] = np.log1p(df_feat["vendor_txn_count"])

    # historical primary slab for each vendor (mode)
    slab_mode = df_feat.groupby("vendor_id")["gst_slab"].agg(lambda x: x.mode().iloc[0])
    df_feat["vendor_primary_slab"] = df_feat["vendor_id"].map(slab_mode)
else:
    df_feat["vendor_txn_count_log"] = 0.0
    df_feat["vendor_primary_slab"] = "UNKNOWN"

df_feat.head()


,transaction_id,transaction_date,amount,currency,vendor_id,gst_applicable,gst_slab,itc_eligible,is_anomaly,txn_month,txn_quarter,txn_dow,financial_year,amount_log,amount_bucket,vendor_txn_count,vendor_txn_count_log,vendor_primary_slab
0,NTX0000001,2024-04-01,2016.36,INR,V0033,True,5%,FALSE,0,4,2,0,2024,7.609545,1k-5k,110,4.709530,5%
1,NTX0000002,2024-04-01,8530.60,INR,V0005,True,18%,TRUE,0,4,2,0,2024,9.051532,5k-20k,146,4.990433,18%
2,NTX0000003,2024-04-01,33181.64,INR,V0014,True,18%,TRUE,0,4,2,0,2024,10.409782,20k+,200,5.303305,18%
3,NTX0000004,2024-04-01,14510.68,INR,V0004,True,18%,TRUE,0,4,2,0,2024,9.582709,5k-20k,201,5.308268,18%
4,NTX0000005,2024-04-01,4667.38,INR,V0021,True,5%,FALSE,0,4,2,0,2024,8.448567,1k-5k,95,4.564348,5%


In [5]:
# Cell 5: Define features and target

# Drop identifier from features
ID_COL = "transaction_id"
TARGET_COL = "gst_slab"

NUM_FEATURES = ["amount", "amount_log", "vendor_txn_count_log"]
CAT_FEATURES = [
    "amount_bucket",
    "txn_month",
    "txn_quarter",
    "txn_dow",
    "financial_year",
    "vendor_id",
    "vendor_primary_slab",
]

X = df_feat[NUM_FEATURES + CAT_FEATURES].copy()
y = df_feat[TARGET_COL].copy()

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())


X shape: (5000, 10)
y distribution:
gst_slab
18%    3418
5%      818
12%     583
0%      166
28%      15
Name: count, dtype: int64


In [6]:
# Cell 6: Train/test split and label encoding

le = LabelEncoder()
y_enc = le.fit_transform(y)
class_labels = le.classes_
print("Classes:", class_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=RANDOM_STATE,
)

X_train.shape, X_test.shape


Classes: ['0%' '12%' '18%' '28%' '5%']


((4000, 10), (1000, 10))

In [7]:
# Cell 7: Preprocessor and model pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUM_FEATURES),
        ("cat", categorical_transformer, CAT_FEATURES),
    ]
)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", xgb_clf),
    ]
)

pipe


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['amount', 'amount_log',
                                                   'vendor_txn_count_log']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['amount_bucket', 'txn_month',
                                                   'txn_quarter', 'txn_dow',
                                                   'financial_year',
                                                   'vendor_id',
                                                   'vendor_primary_slab'])])),
                ('c...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [8]:
# Cell 8: Train and evaluate

pipe.fit(X_train, y_train)

y_pred_enc = pipe.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)
y_test_str = le.inverse_transform(y_test)

print("Accuracy:", accuracy_score(y_test_str, y_pred))
print("Macro F1:", f1_score(y_test_str, y_pred, average="macro"))
print("\nClassification report:")
print(classification_report(y_test_str, y_pred))


Accuracy: 0.988
Macro F1: 0.77686056386623

Classification report:
              precision    recall  f1-score   support

          0%       1.00      0.85      0.92        33
         12%       0.99      0.98      0.99       117
         18%       0.99      1.00      0.99       683
         28%       0.00      0.00      0.00         3
          5%       0.97      1.00      0.98       164

    accuracy                           0.99      1000
   macro avg       0.79      0.77      0.78      1000
weighted avg       0.99      0.99      0.99      1000



c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

In [9]:
# Cell 9: Confidence scoring & inference helper

from typing import Dict, Any

CONFIDENCE_THRESHOLD = 0.75

def predict_gst_slab_row(row: pd.Series) -> Dict[str, Any]:
    """
    row: single transaction row from df_feat (with engineered features)
    """
    X_row = row[NUM_FEATURES + CAT_FEATURES].to_frame().T

    proba = pipe.predict_proba(X_row)[0]
    idx = int(np.argmax(proba))
    slab_pred = le.inverse_transform([idx])[0]
    confidence = float(proba[idx])

    return {
        "predicted_gst_slab": slab_pred,
        "confidence_score": confidence,
    }

# quick test on one row
sample_row = df_feat.iloc[0]
predict_gst_slab_row(sample_row)


{'predicted_gst_slab': '5%', 'confidence_score': 0.9997144341468811}

In [10]:
# Cell 10: Rule engine for tax_category and itc_eligible

def map_slab_to_tax_category(slab: str) -> str:
    s = str(slab).strip().lower()
    if s in ["0%", "exempt", "nil"]:
        return "exempt"
    if s == "5%":
        return "reduced"
    if s == "12%":
        return "standard_lower"
    if s == "18%":
        return "standard"
    if s == "28%":
        return "higher"
    return "unknown"


def compute_itc_eligibility(slab: str, gst_applicable: bool, context: dict = None) -> bool:
    """
    Conservative ITC rule:
    - If GST not applicable or slab is exempt/0%, no ITC.
    - Otherwise ITC allowed (can be refined with more context).
    """
    if not gst_applicable:
        return False

    s = str(slab).strip().lower()
    if s in ["0%", "exempt", "nil"]:
        return False

    # Future: add blocked credit / expense-type rules via context
    return True

# quick test
print(map_slab_to_tax_category("18%"), compute_itc_eligibility("18%", True))
print(map_slab_to_tax_category("exempt"), compute_itc_eligibility("0%", True))


standard True
exempt False


In [11]:
# Cell 11: Anomaly detection model (Isolation Forest)

# Use a subset of numeric features for anomaly detection
ANOM_NUM_FEATURES = ["amount", "amount_log", "vendor_txn_count_log"]

X_num = df_feat[ANOM_NUM_FEATURES].values

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=RANDOM_STATE,
)
iso.fit(X_num)

def flag_anomaly(row: pd.Series, threshold: float = -0.2) -> bool:
    X_row = row[ANOM_NUM_FEATURES].to_frame().T.values
    score = iso.decision_function(X_row)[0]
    return score < threshold


In [12]:
# Cell 12: End-to-end prediction for a single transaction

def predict_transaction_full(row: pd.Series) -> dict:
    base = predict_gst_slab_row(row)
    slab_pred = base["predicted_gst_slab"]
    confidence = base["confidence_score"]

    anomaly_flag = flag_anomaly(row)

    gst_applicable = bool(row.get("gst_applicable", True))
    tax_category = map_slab_to_tax_category(slab_pred)
    itc_eligible = compute_itc_eligibility(slab_pred, gst_applicable)

    needs_review = (confidence < CONFIDENCE_THRESHOLD) or anomaly_flag

    return {
        "transaction_id": row[ID_COL],
        "predicted_gst_slab": slab_pred,
        "confidence_score": confidence,
        "needs_review": bool(needs_review),
        "tax_category": tax_category,
        "itc_eligible": bool(itc_eligible),
        "anomaly_flag": bool(anomaly_flag),
    }

# Example on first 3 rows
for i in range(3):
    print(predict_transaction_full(df_feat.iloc[i]))


{'transaction_id': 'NTX0000001', 'predicted_gst_slab': '5%', 'confidence_score': 0.9997144341468811, 'needs_review': False, 'tax_category': 'reduced', 'itc_eligible': True, 'anomaly_flag': False}
{'transaction_id': 'NTX0000002', 'predicted_gst_slab': '18%', 'confidence_score': 0.998225748538971, 'needs_review': False, 'tax_category': 'standard', 'itc_eligible': True, 'anomaly_flag': False}
{'transaction_id': 'NTX0000003', 'predicted_gst_slab': '18%', 'confidence_score': 0.9977999329566956, 'needs_review': False, 'tax_category': 'standard', 'itc_eligible': True, 'anomaly_flag': False}


In [13]:
# Cell 13: Batch prediction for all transactions

outputs = []
for _, row in df_feat.iterrows():
    outputs.append(predict_transaction_full(row))

pred_df = pd.DataFrame(outputs)
pred_df.head()


,transaction_id,predicted_gst_slab,confidence_score,needs_review,tax_category,itc_eligible,anomaly_flag
0,NTX0000001,5%,0.999714,False,reduced,True,False
1,NTX0000002,18%,0.998226,False,standard,True,False
2,NTX0000003,18%,0.997800,False,standard,True,False
3,NTX0000004,18%,0.998804,False,standard,True,False
4,NTX0000005,5%,0.997561,False,reduced,True,False


In [ ]:
# Cell 14: Save model and encoders (optional, for production reuse)

# import joblib

# joblib.dump(pipe, "models/nontext_gst_slab_pipe.joblib")
# joblib.dump(le, "models/nontext_gst_slab_label_encoder.joblib")
# joblib.dump(iso, "models/nontext_gst_slab_iso_anomaly.joblib")

# pred_df.to_csv("outputs/nontext_gst_slab_predictions.csv", index=False)
# print("Saved model artifacts and predictions.")
